# Screen sensitivity to the classical kernel, and F1 learning curves

For every saved kernel file in Drive, recomputes (1) classical kernel-target alignment under
several classical kernels, linear, polynomial degree 3, and RBF at gamma in {0.5/d, 1/d, 2/d, scale},
to test whether the screen depends on the kernel used for A_C, and (2) F1 learning curves for the
quantum, linear, and RBF SVMs at N in {50,100,200,400,700,1000} with a consistent metric.
CPU only, about ten seconds per file. Output: `kernel_choice_and_curves.json`.

In [ ]:
import numpy as np, json, os, glob, re
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel, polynomial_kernel
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive'); SAVE_DIR='/content/drive/MyDrive/qml_project'

def kta(K,y):
    y=np.asarray(y); n=len(y); Y=np.zeros((n,n))
    for c in np.unique(y): m=(y==c).astype(float); Y+=np.outer(m,m)
    H=np.eye(n)-np.ones((n,n))/n; Kc=H@K@H; Yc=H@Y@H
    return float(np.sum(Kc*Yc)/np.sqrt(np.sum(Kc*Kc)*np.sum(Yc*Yc)))

files=sorted(set(glob.glob(os.path.join(SAVE_DIR,'*kernel*.npz')))); print(f"found {len(files)} kernel files"); out={}
for f in files:
    z=np.load(f); keys=set(z.files)
    kK=next((k for k in ['K_train_quantum','K_train'] if k in keys),None); kT=next((k for k in ['K_test_quantum','K_test'] if k in keys),None)
    kX=next((k for k in ['X_tr','X_train','X_tr_subset'] if k in keys),None); kY=next((k for k in ['y_tr','y_train','y_tr_subset'] if k in keys),None)
    kXt=next((k for k in ['X_te','X_test','X_te_subset'] if k in keys),None); kYt=next((k for k in ['y_te','y_test','y_te_subset'] if k in keys),None)
    if not (kK and kX and kY): print(f"  skip {os.path.basename(f)}"); continue
    K=z[kK].astype(np.float64); X=z[kX].astype(np.float64); y=z[kY]; d=X.shape[1]
    tag=re.sub(r'_?kernel.*$','',os.path.basename(f)).replace('.npz','')
    rec={'tag_guess':tag,'n':int(K.shape[0])}
    # (1) alignment under alternative classical kernels
    var=X.var(); rec['A_C_variants']={
        'linear':kta(linear_kernel(X,X),y),
        'poly3':kta(polynomial_kernel(X,X,degree=3,coef0=1,gamma=1.0/d),y),
        'rbf_0.5/d':kta(rbf_kernel(X,X,gamma=0.5/d),y),
        'rbf_1/d':kta(rbf_kernel(X,X,gamma=1.0/d),y),
        'rbf_2/d':kta(rbf_kernel(X,X,gamma=2.0/d),y),
        'rbf_scale':kta(rbf_kernel(X,X,gamma=1.0/(d*var)),y)}
    # (2) learning curves, F1 with a consistent averaging rule
    # test split: from the kernel file if present, else from the matching embeddings file
    Kt=z[kT].astype(np.float64) if kT else None; Xt=z[kXt].astype(np.float64) if kXt else None; yt=z[kYt] if kYt else None
    if Kt is not None and (Xt is None or yt is None):
        cand=[os.path.join(SAVE_DIR,f'{tag}_embeddings.npz'), os.path.join(SAVE_DIR,'embeddings_week2.npz') if tag=='' else '']
        for e in cand:
            if e and os.path.exists(e):
                ze=np.load(e); kx=next((k for k in ['X_te_subset','X_test','X_te'] if k in ze.files),None); ky=next((k for k in ['y_te_subset','y_test','y_te'] if k in ze.files),None)
                if kx and ky and len(ze[ky])==Kt.shape[0]: Xt=ze[kx].astype(np.float64); yt=ze[ky]; print(f"    test split from {os.path.basename(e)}"); break
    if Kt is not None and Xt is not None and yt is not None:
        avg='binary' if len(np.unique(y))==2 else 'macro'
        f1=lambda a,b: float(f1_score(a,b,average=avg))
        sizes=[50,100,200,400,700,1000]; lc={'quantum':[],'linear':[],'rbf':[]}
        for n in sizes:
            lc['quantum'].append(f1(yt,SVC(kernel='precomputed',C=1.0).fit(K[:n,:n],y[:n]).predict(Kt[:,:n])))
            lc['linear'].append(f1(yt,SVC(kernel='linear',C=1.0,random_state=42).fit(X[:n],y[:n]).predict(Xt)))
            lc['rbf'].append(f1(yt,SVC(kernel='rbf',C=1.0,gamma='scale',random_state=42).fit(X[:n],y[:n]).predict(Xt)))
        rec['learning_curve']={'sizes':sizes,**lc,'metric':avg}
    else:
        rec['learning_curve']=None; print(f"  {os.path.basename(f)}: no test split saved, curve skipped")
    out[os.path.basename(f)]=rec
    print(f"  {os.path.basename(f):<45} A_C rbf1/d={rec['A_C_variants']['rbf_1/d']:.4f} linear={rec['A_C_variants']['linear']:.4f} poly={rec['A_C_variants']['poly3']:.4f}")
p=os.path.join(SAVE_DIR,'kernel_choice_and_curves.json'); json.dump(out,open(p,'w'),indent=2); print(f"\nsaved {p} ({len(out)} entries). Upload it.")